In [ ]:
# Chcecking for GPU and CUDA version

import torch
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device : {device}')

In [ ]:
# Importing necessary libraries

import torch.nn as nn
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
import torch.optim as optim
import os

In [ ]:
# importing the ResNet50 model 

model = models.resnet50(weights = models.ResNet50_Weights.DEFAULT)

# Unfreezing the last layer only

In [ ]:
# Freezing the layers of the model
for param in model.parameters():
    param.requires_grad = False

In [ ]:
# Modifying the final fully connected layer of the model to match the number of classes of Brain Tumor Dataset

num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(p = 0.3),
    nn.Linear(num_features,4)
)    # 4 classes in the brain tumor dataset

In [ ]:
# Providing model to the device

model = model.to(device)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])
])

In [ ]:
# Importing the Dataset and creating dataloaders

base_dir = '.'

train_dataset = datasets.ImageFolder(root=os.path.join(base_dir, 'Datasets', 'Training'), transform = transform)
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)

val_dataset = datasets.ImageFolder(root=os.path.join(base_dir, 'Datasets', 'Testing'), transform = transform)
val_loader = DataLoader(val_dataset, batch_size = 32, shuffle = False)

In [ ]:
# Defining the criterion and optimizer

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr = 0.0001)

In [ ]:
# Defining the training loop for the model
epochs = 100

for epoch in range(epochs):
    model.train()
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'Epoch {epoch+1}/{epochs}, Loss : {loss.item(): .4f}')

In [ ]:
# Validating the model on the validation dataset

model.eval()

correct = 0
total = 0
val_loss = 0.0

with torch.no_grad():
    
    for images, labels in val_loader:
        images,labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        val_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
accuracy = 100 * correct/total
avg_loss = val_loss/len(val_loader)

print(f'Validation Accuracy : {accuracy: .2f}%')
print(f'Validation Loss : {avg_loss: .4f}')

In [ ]:
# Classes of Brain Tumor in the Dataset

class_names = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']

Analayzing the Classification by the model

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

In [ ]:
all_preds = []
all_labels = []

model.eval()

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs,1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
# Creating the Confusion Matrix

cm = confusion_matrix(all_labels, all_preds)
class_names = train_dataset.classes


#Plotting using Seaborn

plt.figure(figsize = (10,8))
sns.heatmap(cm, annot = True, fmt = 'd', cmap = 'Blues',
            xticklabels = class_names, yticklabels = class_names)
plt.xlabel('Predicted Class')
plt.ylabel('Actual Class')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Generating the classification report

from sklearn.metrics import classification_report

report = classification_report(all_labels, all_preds, target_names = class_names)

print('Classification Report: \n', report)